In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Generate SYnthetic .npy files for each class
import os
import numpy as np
import tensorflow as tf

# Load the generator model
generator = tf.keras.models.load_model('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/generator.h5')

# Define the save directory
save_dir = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset'
os.makedirs(save_dir, exist_ok=True)

# Define the function to generate synthetic genomaps and save as .npy
def generate_and_save_npy(generator, num_samples, label, latent_dim=128, save_path=None):
    """
    Generate synthetic genomaps using the generator and save them as a .npy file.

    Parameters:
    - generator: Trained GAN generator model.
    - num_samples: Number of synthetic images to generate.
    - label: The conditional label for the generator (e.g., -0.5 for Low, 0.5 for High).
    - latent_dim: Dimensionality of the latent space (default: 128).
    - save_path: Path to save the generated .npy file (default: None).
    """
    # Generate random noise (latent vectors)
    _noise = tf.random.normal(shape=(num_samples, latent_dim))

    # Create labels array
    labels = np.array([label] * num_samples).reshape((num_samples, 1))

    # Generate synthetic images
    synthetic_images = generator.predict([_noise, labels])

    # Save the synthetic images as a .npy file
    if save_path:
        np.save(save_path, synthetic_images)
        print(f"Synthetic genomaps saved to {save_path}")
    else:
        print("Save path is not provided!")

# Generate and save synthetic genomaps for High and Low classes
#b = low; mono = high
#Y = [-0.5] * low_data.shape[0] +  [0.5] * high_data.shape[0]
generate_and_save_npy(generator, num_samples=1186, label=0.5, save_path=os.path.join(save_dir, 'synthetic_mono.npy'))
generate_and_save_npy(generator, num_samples=1186, label=-0.5, save_path=os.path.join(save_dir, 'synthetic_b.npy'))


38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
Synthetic genomaps saved to /content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/synthetic_mono.npy
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Synthetic genomaps saved to /content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/synthetic_b.npy


#b = low; mono = high

In [ ]:
import numpy as np

# Path to the .npy file
file_path = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/synthetic_b.npy'

# Load the .npy file
synthetic_high = np.load(file_path)

# Print the shape (dimensions)
print(f"Shape of synthetic_high.npy: {synthetic_high.shape}")


Shape of synthetic_high.npy: (1186, 40, 40, 1)


In [ ]:
#Reverse Gene Expression

In [ ]:
import numpy as np
import pandas as pd

# Load the saved transformation matrix T
T = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/Reverse GeneExpression Required Docs/transformation_matrix_T_mono_class.npy')  # Update with the correct path
num_genes = T.shape[1]  # Number of genes in the original dataset

# Load original means and standard deviations
original_means = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/Reverse GeneExpression Required Docs/original_means_mono_class.npy')  # Update with the correct path
original_stds = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/Reverse GeneExpression Required Docs/original_stds_mono_class.npy')  # Update with the correct path

# Load synthetic genomap data
synthetic_genomap_path = r'/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/synthetic_mono.npy'  # Update with the correct path
synthetic_genomaps = np.load(synthetic_genomap_path)  # Shape: (num_samples, rowNum, colNum, 1)

# Inspect dimensions
print(f"Synthetic genomaps shape: {synthetic_genomaps.shape}")

# Function to reverse the genomap to recover the original data
def reverse_genomap(genomaps, T, num_genes):
    num_samples, rowNum, colNum, _ = genomaps.shape
    totalGridPoint = rowNum * colNum

    # Scale T to match grid points
    projMat = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)  # Pseudoinverse of T

    # Flatten genomaps and project back to gene expression
    projM = np.zeros((num_samples, num_genes))
    for i in range(num_samples):
        ex = genomaps[i, :, :, 0].flatten(order='F')  # Flatten in column-major order
        projM[i, :] = ex[:num_genes]

    # Reverse the projection to obtain the original data
    data = np.matmul(projM, projMat_inv)
    return data

# Reverse the synthetic genomaps
reversed_genomap = reverse_genomap(synthetic_genomaps, T, num_genes)

# Apply inverse normalization to the recovered data
recovered_data = (reversed_genomap * original_stds) + original_means

# Optionally set very small values close to zero (comment out if unnecessary)
#recovered_data[np.abs(recovered_data) < 1e-10] = 0  # Comment out this line if it causes issues

# Convert the recovered gene expression data into a pandas DataFrame
recovered_data_df = pd.DataFrame(recovered_data)

# Save the recovered gene expression values to a CSV file
recovered_file_path = r'/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/recovered_gene_expression_mono_class.csv'  # Update the path
recovered_data_df.to_csv(recovered_file_path, index=False, header=False)

print(f"Recovered gene expression data saved to: {recovered_file_path}")


Synthetic genomaps shape: (1186, 40, 40, 1)
Recovered gene expression data saved to: /content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/recovered_gene_expression_mono_class.csv


With MAX and MIN value

In [ ]:
import numpy as np
import pandas as pd

# Load the saved transformation matrix T
T = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/Reverse GeneExpression Required Docs/transformation_matrix_T_b_class.npy')  # Update with the correct path
num_genes = T.shape[1]  # Number of genes in the original dataset

# Load original means and standard deviations
original_means = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/Reverse GeneExpression Required Docs/original_means_b_class.npy')  # Update with the correct path
original_stds = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/Reverse GeneExpression Required Docs/original_stds_b_class.npy')  # Update with the correct path

# Load synthetic genomap data
synthetic_genomap_path = r'/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/synthetic_b.npy'  # Update with the correct path
synthetic_genomaps = np.load(synthetic_genomap_path)  # Shape: (num_samples, rowNum, colNum, 1)

# Define min and max values for reverse normalization
_MIN = -2.4649  # Replace with the actual min value
_MAX = 34.409   # Replace with the actual max value

# Reverse normalization for synthetic genomaps, for tanh purpose data = -1 + 2*(data - _MIN)/(_MAX - _MIN) was used
synthetic_genomaps = ((synthetic_genomaps + 1) * (_MAX - _MIN) / 2) + _MIN

# Inspect dimensions
print(f"Synthetic genomaps shape after reverse normalization: {synthetic_genomaps.shape}")

# Function to reverse the genomap to recover the original data
def reverse_genomap(genomaps, T, num_genes):
    num_samples, rowNum, colNum, _ = genomaps.shape
    totalGridPoint = rowNum * colNum

    # Scale T to match grid points
    projMat = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)  # Pseudoinverse of T

    # Flatten genomaps and project back to gene expression
    projM = np.zeros((num_samples, num_genes), dtype=np.float32)
    for i in range(num_samples):
        ex = genomaps[i, :, :, 0].flatten(order='F')  # Flatten in column-major order
        projM[i, :] = ex[:num_genes]

    # Reverse the projection to obtain the original data
    data = np.matmul(projM, projMat_inv)
    return data

# Reverse the synthetic genomaps
reversed_genomap = reverse_genomap(synthetic_genomaps, T, num_genes)

# Apply inverse normalization to the recovered data
recovered_data = (reversed_genomap * original_stds) + original_means

# Ensure float32 dtype for the recovered data
recovered_data = recovered_data.astype(np.float32)

# Convert the recovered gene expression data into a pandas DataFrame
recovered_data_df = pd.DataFrame(recovered_data)

# Save the recovered gene expression values to a CSV file
recovered_file_path = r'/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/recovered_gene_expression_b_class.csv'  # Update the path
recovered_data_df.to_csv(recovered_file_path, index=False, header=False, float_format='%.6f')

print(f"Recovered gene expression data saved to: {recovered_file_path}")


Synthetic genomaps shape after reverse normalization: (1186, 40, 40, 1)
Recovered gene expression data saved to: /content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/recovered_gene_expression_b_class.csv
